Các bước thực hiện trong colab này:
+ Đọc dữ liệu
+ Clean data
+ Cho vào GAN
+ Chia dữ liệu train, test
+ Scale tập train, test
+ Cho vào mô hình
- tỉ lệ trên tập test cân bằng là 95.5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1115
1.0,846
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1655
2.0,295
3.0,176


Clean data

In [ ]:
data.duplicated().sum()

np.int64(13)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.isna().sum().sum()

np.int64(0)

GAN

In [ ]:
import pandas as pd
from ctgan import CTGAN
from sklearn.model_selection import train_test_split

target_col = 'fetal_health'

print("Kích thước data gốc:", data.shape)
display(data.head())

print("\nPhân bố lớp ban đầu:")
print(data[target_col].value_counts().sort_index())

# =============================
# 2. Train CTGAN trên toàn bộ data
#    fetal_health vẫn giữ dạng số
# =============================
gan_df = data.copy()

ctgan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan.fit(gan_df, discrete_columns=[target_col])

print("\nĐã train xong CTGAN")

# =============================
# 3. Sinh thêm dữ liệu để cân bằng
# =============================
class_counts = gan_df[target_col].value_counts()
max_count = class_counts.max()

synthetic_parts = []

for cls, count in class_counts.items():
    need = max_count - count

    if need <= 0:
        continue

    print(f"\nLớp {cls} cần sinh thêm {need} mẫu")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 2)
        fake_batch = ctgan.sample(sample_n)

        # vì target là số, lọc trực tiếp theo số
        fake_cls = fake_batch[fake_batch[target_col] == cls].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_parts.append(fake_cls_final)

# =============================
# 4. Gộp dữ liệu thật + synthetic
# =============================
if len(synthetic_parts) > 0:
    synthetic_df = pd.concat(synthetic_parts, axis=0).reset_index(drop=True)
    balanced_df = pd.concat([gan_df, synthetic_df], axis=0).reset_index(drop=True)
else:
    balanced_df = gan_df.copy()

print("\nPhân bố lớp sau CTGAN:")
print(balanced_df[target_col].value_counts().sort_index())

# =============================
# 5. Tách X, y
# =============================
X = balanced_df.drop(columns=[target_col]).copy()
y = balanced_df[target_col].astype(int)

# =============================
# 6. Chia train/test
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nKích thước tập train/test:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPhân bố y_train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố y_test:")
print(y_test.value_counts().sort_index())

Kích thước data gốc: (2113, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1.0    1646
2.0     292
3.0     175
Name: count, dtype: int64


Gen. (-00.26) | Discrim. (+00.05): 100%|██████████| 300/300 [07:13<00:00,  1.44s/it]



Đã train xong CTGAN

Lớp 2.0 cần sinh thêm 1354 mẫu
Đã lấy được 879/1354
Đã lấy được 1751/1354

Lớp 3.0 cần sinh thêm 1471 mẫu
Đã lấy được 864/1471
Đã lấy được 1670/1471

Phân bố lớp sau CTGAN:
fetal_health
1.0    1646
2.0    1646
3.0    1646
Name: count, dtype: int64

Kích thước tập train/test:
X_train: (3950, 21)
X_test : (988, 21)

Phân bố y_train:
fetal_health
1    1317
2    1317
3    1316
Name: count, dtype: int64

Phân bố y_test:
fetal_health
1    329
2    329
3    330
Name: count, dtype: int64


Scale data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo scaler
scaler = MinMaxScaler()

# Fit trên train, transform cả train và test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Nếu muốn giữ DataFrame
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("Đã scale dữ liệu xong")
display(X_train_scaled.head())

Đã scale dữ liệu xong


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1251,0.256997,0.338380,0.010631,0.597539,0.208109,0.439865,0.361899,0.135765,0.219850,0.071884,...,0.463672,0.268078,0.234121,0.108991,0.103892,0.626353,0.479795,0.502183,0.045960,0.972908
2069,0.414627,0.161948,0.026466,0.547231,0.159752,0.439865,0.361899,0.740539,0.372201,0.071884,...,0.208784,0.709747,0.281736,0.108991,0.004325,0.650855,0.518852,0.549086,0.019350,0.105376
4320,0.467564,0.248473,0.006968,0.157680,0.101876,0.487312,0.361533,0.809150,0.085388,0.418676,...,0.425136,0.676296,0.237814,0.088507,0.003548,0.501325,0.461787,0.523397,0.025041,0.538187
3151,0.503908,0.436469,0.010662,0.393182,0.120439,0.810019,0.497350,0.389890,0.237271,0.370098,...,0.471412,0.555309,0.440320,0.267855,0.004063,0.899814,0.685880,0.728186,0.018711,0.956566
187,0.529267,0.250164,0.010631,0.547231,0.159752,0.439865,0.361899,0.382612,0.150600,0.213764,...,0.299057,0.750643,0.456323,0.475877,0.203460,0.846866,0.737574,0.767969,0.045960,0.539142


RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Khởi tạo model
rf_model_gan = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model_gan.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model_gan.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision weighted:", precision_score(y_test, y_pred, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred, average='macro'))
print("\nDetail about one class:")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9554655870445344
Precision weighted: 0.9552593073402571
Recall weighted: 0.9554655870445344
F1 weighted: 0.9552901942724121
Precision macro: 0.9552661710270405
Recall macro: 0.9554726597279789
F1 macro: 0.9552970892529467

Detail about one class:

Classification Report:
               precision    recall  f1-score   support

           1       0.97      0.99      0.98       329
           2       0.94      0.92      0.93       329
           3       0.95      0.95      0.95       330

    accuracy                           0.96       988
   macro avg       0.96      0.96      0.96       988
weighted avg       0.96      0.96      0.96       988


Confusion Matrix:
 [[327   2   0]
 [  8 304  17]
 [  1  16 313]]


XgBoot

In [ ]:
y_train = y_train.astype(int) - 1
y_test = y_test.astype(int) - 1

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Khởi tạo model XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',   # dùng cho phân loại nhiều lớp
    num_class=len(y_train.unique()),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

# Train model
xgb_model.fit(X_train_scaled, y_train)

# Predict
y_pred_xg = xgb_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred_xg))
print("Precision weighted:", precision_score(y_test, y_pred_xg, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred_xg, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred_xg, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred_xg, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred_xg, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred_xg, average='macro'))

print("\nDetail about one class:")
print("Classification Report:\n", classification_report(y_test, y_pred_xg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xg))

Accuracy: 0.9625506072874493
Precision weighted: 0.9624419181944857
Recall weighted: 0.9625506072874493
F1 weighted: 0.9624323051790354
Precision macro: 0.9624413902869188
Recall macro: 0.9625679285253753
F1 macro: 0.9624407184159273

Detail about one class:
Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       329
           1       0.95      0.95      0.95       329
           2       0.96      0.95      0.95       330

    accuracy                           0.96       988
   macro avg       0.96      0.96      0.96       988
weighted avg       0.96      0.96      0.96       988


Confusion Matrix:
 [[328   1   0]
 [  6 311  12]
 [  2  16 312]]
